# 🏥 ChakraModel Cloud GPU Evaluation Notebook (Kaggle/Colab)

This notebook contains the complete environment to load the ChakraModel architecture, load all pretrained weights, and run cross-verification benchmarks using a cloud GPU like Kaggle T4x2 or Google Colab.

## Setup Instructions for Kaggle:
1. **Upload your local `chakramodel` folder as a Kaggle Dataset**. 
   - Create a ZIP file of your local `chakramodel` directory (ensure `weights/best.pt` is inside).
   - In Kaggle, click **Add Input** -> **Upload a Dataset**.
   - Name it `chakramodel-data`.
   - Your files will then be available at `/kaggle/input/chakramodel-data/`.
2. In the Kaggle Notebook Settings (right sidebar), turn on **Accelerator: GPU P100 or T4x2**.
3. Set `REPO_PATH` in the cell below to point to your dataset (e.g. `/kaggle/input/chakramodel-data/chakramodel`).


In [2]:
!apt-get update && apt-get install -y ffmpeg
!pip install ultralytics opencv-python-headless pandas numpy torch torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.0 MB/s eta 0:00:00


In [3]:
import sys
import os
from pathlib import Path

def find_repo_root():
    """Automatically finds the dataset folder in Kaggle by searching for the 'src' directory"""
    if os.path.exists('/kaggle/input'):
        for path in Path('/kaggle/input').rglob('src'):
            if path.is_dir():
                return str(path.parent)
    return '.' # Fallback if running locally

REPO_PATH = find_repo_root()

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print(f"Working directory set to: {os.getcwd()}")

Working directory set to: /kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle


In [ ]:
import sys
sys.path.append(REPO_PATH)
from src.chakranet_segmenter import ChakraNet

# Intercept the initialization in memory to force 352x352 resolution
original_init = ChakraNet.__init__

def patched_init(self, device=None, img_size=(352, 352), weights_path=None):
    original_init(self, device=device, img_size=(352, 352), weights_path=weights_path)

ChakraNet.__init__ = patched_init
print('✅ Monkey-patched ChakraNet! Full-resolution benchmarking enabled.')

In [4]:
import torch
from ultralytics import YOLO
from src.chakranet_segmenter import ChakraNet
from src.paris_classifier import ParisClassifier
from src.benchmark_kvasir import run_benchmark
from pathlib import Path

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
CUDA Available: True
Device Name: Tesla T4


## Download Evaluation Dataset
Downloading the Kvasir-SEG benchmark dataset to verify the cross-dataset metrics and test inference.

In [5]:
from src.download_kvasir import download_kvasir_seg

# This will download the dataset to REPO_PATH/data/kvasir-seg 
# (If it fails due to network, it falls back to synthetic data so pipelines can still run)
root_dir = Path(REPO_PATH)

# Note: In Kaggle /kaggle/input is read-only. If the dataset needs to be written to,
# we must point root_dir to /kaggle/working. But download_kvasir_seg defaults to root_dir / "data".
if "/kaggle/input" in str(root_dir):
    working_dir = Path("/kaggle/working")
    dataset_path = download_kvasir_seg(working_dir)
else:
    working_dir = root_dir
    dataset_path = download_kvasir_seg(root_dir)

print(f"Dataset ready at: {dataset_path}")

  Progress: 100.0%  (44MB / 44MB)
[EXTRACT] kvasir-seg.zip...
[OK] kvasir-seg ready: 1000 images
Dataset ready at: /kaggle/working/data/kvasir-seg


## Load the Weights
Loading the trained YOLO weights and initializing the ChakraModel PraNet architecture.

In [6]:
# Determine the path to YOLO best weights
weight_candidates = [
    root_dir / "weights/best.pt",
    root_dir / "outputs/polyp_yolov8x/weights/best.pt",
    root_dir / "best.pt",
]

best_weight_path = None
for c in weight_candidates:
    if c.exists():
        best_weight_path = c
        break

if best_weight_path:
    print(f"Loading YOLO weights from: {best_weight_path}")
    yolo_model = YOLO(str(best_weight_path))
    if torch.cuda.is_available():
        yolo_model.to('cuda')
else:
    print("WARNING: YOLO best.pt weights not found in standard directories. Make sure weights are uploaded.")

# Initialize PraNet
print("Initializing PraNet Segmenter...")
pranet = ChakraNet(device='cuda' if torch.cuda.is_available() else 'cpu')

# Initialize Paris Classifier
print("Initializing Paris Classifier...")
paris = ParisClassifier()
print("All ChakraModel components loaded successfully!")

Loading YOLO weights from: /kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/weights/best.pt
Initializing PraNet Segmenter...
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 173MB/s]


[INFO] ChakraNet: Loaded weights from /kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/weights/pranet_kvasir_best.pth
Initializing Paris Classifier...
All ChakraModel components loaded successfully!


## Run Benchmark Evaluation
Run the benchmark script directly on Kvasir-SEG to calculate Dice, mIoU, Sensitivity, Specificity, and FPS. This will automatically cross-verify the results from local training.

In [ ]:
out_dir = working_dir / "outputs" / "eval_kaggle"
out_dir.mkdir(parents=True, exist_ok=True)

print("Running full benchmark on Kvasir-SEG...")
# Temporarily adjust dataset paths since Kaggle /kaggle/input is read-only and 
# download_kvasir_seg puts it in working_dir.
import src.benchmark_kvasir as bmk
# Override the dataset root to point to the working dir where we downloaded it
metrics = bmk.run_benchmark("kvasir-seg", working_dir, out_dir)

import json
print("\nBenchmark Results Summary:")
print(json.dumps({
    "Dice (DSC)": metrics['dice'],
    "mIoU": metrics['iou'],
    "Sensitivity": metrics['sensitivity'],
    "FPS": metrics['fps']
}, indent=2))

## Test on Live Video (Optional)
If you have `test_input.mp4` uploaded with your dataset, you can process it through the pipeline just like the Gradio app.

In [ ]:
from app import process_video
import glob
from pathlib import Path
import shutil
import os

# 1. Automatically find any .mp4 file in your dataset
mp4_files = list(Path(REPO_PATH).rglob('*.mp4'))
if not mp4_files:
    print('No .mp4 files found in your dataset. Upload one to test the video pipeline.')
else:
    video_file = mp4_files[0]
    print(f'Found video: {video_file}')
    print(f'Processing video (this may take a moment)...')
    
    # Run the ChakraModel tracker
    out_video_path, df_timeline = process_video(
        str(video_file), 
        use_persistence=True, 
        window_size=10, 
        persistence_threshold=0.6, 
        doubt_policy='Warn Only', 
        imgsz_val='1024'
    )
    
    # 2. Safely handle the Kaggle output environment
    working_dir = Path('/kaggle/working') if os.path.exists('/kaggle/working') else Path('.')
    final_vid = working_dir / f'output_{video_file.name}'
    
    try:
        shutil.copy2(out_video_path, final_vid)
        print(f'Processed video saved to output: {final_vid}')
        print('\nTimeline of Detections:')
        display(df_timeline)
    except Exception as e:
        print(f'Error copying to output: {e}')